# Project Vyapaar: Business Data Treasure Hunt (Full Analysis)

**Team:** ____________________

| Member | Role |
|---|---|
| ________ | Data Analyst (cleaning) |
| ________ | Business Analyst (sales, profit, discounts, customers) |
| ________ | Operations Analyst (delivery, reviews) |
| ________ | Visualisation & Strategy (charts, hidden mystery, summary) |

**Our approach:** Explore → Question → Analyse → Verify → Visualise → Interpret → Recommend.
For every result we ask: **"So what?"**

**Data we had:** orders, products, reviews and marketing. (`customers.csv` and `delivery.csv` were not in our folder.)

**Contents:** Stage 1 Data Detective → Stage 2 Business Mystery → Stage 3 Customer & Operations → Stage 4 Hidden Mystery → Bonus Marketing → Management Summary

### How to run this notebook
**On Google Colab:**
1. Upload this notebook to Colab (File → Upload notebook).
2. Click **Runtime → Run all**.
3. When the "Choose Files" button appears, select the 4 data files: `orders.csv`, `products.csv`, `reviews.csv`, `marketing.csv`.

**On Jupyter (laptop):** keep the 4 csv files in the same folder as this notebook (or in a `data` folder next to it) and run all cells.

## Step 0: Load the data

In [ ]:
import pandas as pd
import plotly.express as px
import os

# show numbers with 2 decimals and commas
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 30)

In [ ]:
# This small function finds a csv file whether it is in this folder or in a "data" folder
def find_file(name):
    for folder in ["", "data/", "../data/"]:
        if os.path.exists(folder + name):
            return folder + name
    return None

# If the files are not there we are probably on Google Colab, so ask to upload them
if find_file("orders.csv") is None:
    from google.colab import files
    print("Please upload orders.csv, products.csv, reviews.csv and marketing.csv")
    files.upload()

In [ ]:
# encoding="utf-8-sig" removes a hidden character at the start of orders.csv and reviews.csv
orders = pd.read_csv(find_file("orders.csv"), encoding="utf-8-sig")
products = pd.read_csv(find_file("products.csv"))
marketing = pd.read_csv(find_file("marketing.csv"))

# keep_default_na=False stops pandas from turning the word "None" into a blank.
# In reviews, "None" means "no complaint", so we want to keep it.
reviews = pd.read_csv(find_file("reviews.csv"), encoding="utf-8-sig", keep_default_na=False, na_values=[""])

print("orders:", orders.shape)
print("products:", products.shape)
print("reviews:", reviews.shape)
print("marketing:", marketing.shape)

---
# Stage 1: Data Detective 🔎
Before making any recommendation, we first check if we can trust the data.

### 1.1 First look

In [ ]:
orders.head()

In [ ]:
products.head()

In [ ]:
reviews.head()

In [ ]:
marketing.head()

In [ ]:
orders.info()

`order_date` is stored as text (object), not as a date. We will convert it later.

### 1.2 Missing values

In [ ]:
print("ORDERS")
print(orders.isnull().sum())
print()
print("REVIEWS")
print(reviews.isnull().sum())

In [ ]:
# which orders have a missing customer or discount?
orders[orders["customer_id"].isnull() | orders["discount_pct"].isnull()]

In [ ]:
reviews[reviews["rating"].isnull()]

**What we decided:**
- **campaign_id** is blank for 1,472 orders. That is too many to be a mistake. It just means the order did not come from a campaign, so we label it **"No campaign"**.
- **O00300** has no customer. The sale is still real, so we keep it and call the customer **"Unknown"**.
- **O00778** has no discount. We can work it out from the revenue (see 1.6).
- **R01201** has no rating. We keep the row (it still shows a late delivery and an escalation), and we do **not** make up a rating.

### 1.3 Duplicates

In [ ]:
print("Duplicate rows in orders:", orders.duplicated().sum())
print("Duplicate rows in products:", products.duplicated().sum())
print("Duplicate rows in reviews:", reviews.duplicated().sum())
print("Duplicate rows in marketing:", marketing.duplicated().sum())

In [ ]:
orders[orders.duplicated(keep=False)]

In [ ]:
products[products.duplicated(keep=False)]

The duplicates are exact copies (same ID, same values), so they are double entries and we remove them.

⚠️ The product duplicate (**P040**) is important. If we don't remove it, every P040 order will be counted **twice** when we join the tables.

### 1.4 Spelling and category problems

In [ ]:
products["category"].value_counts()

In [ ]:
orders["sales_channel"].value_counts()

In [ ]:
products[(products["category"] == "Electronic") | (products["subcategory"] == "skin care")]

- **P011** is called "Hair Care Item 011", but its category says "Electronic".
- **P074** is called "Small Appliances Item 074", but its subcategory says "skin care".
- "web site" is just another spelling of "Website".

To be sure about the two products, we checked the **mark-up** (list price ÷ cost). Each category seems to have its own fixed mark-up:

In [ ]:
products["markup"] = products["list_price_inr"] / products["unit_cost_inr"]
products.groupby("category")["markup"].mean()

In [ ]:
products[products["product_id"].isin(["P011", "P074"])][["product_id", "product_name", "category", "subcategory", "markup"]]

P011's mark-up (1.92) matches **Beauty**, and P074's (1.47) matches **Home & Kitchen**. So the names were right and the labels were wrong:
- P011 → category = **Beauty**
- P074 → subcategory = **Small Appliances**

### 1.5 Do the money columns add up?
We re-calculated revenue ourselves: **revenue = quantity × price × (1 − discount)**

In [ ]:
check = orders.copy()
check["my_revenue"] = check["quantity"] * check["unit_price_inr"] * (1 - check["discount_pct"])
check["difference"] = (check["my_revenue"] - check["revenue_inr"]).abs()

print("Orders where revenue does not match:", (check["difference"] > 1).sum())
print("Orders where profit is not revenue - cost:", ((check["revenue_inr"] - check["cost_inr"] - check["profit_inr"]).abs() > 1).sum())

Everything matches 👍 so we can trust the revenue, cost and profit numbers.

### 1.6 Working out the missing discount
Since revenue = qty × price × (1 − discount), the missing discount = **1 − revenue ÷ (qty × price)**

In [ ]:
row = orders[orders["discount_pct"].isnull()]
print(1 - row["revenue_inr"] / (row["quantity"] * row["unit_price_inr"]))
print()
print("Discount levels used by the company:", sorted(orders["discount_pct"].dropna().unique().tolist()))

The answer is **0.30**, which is one of the company's normal discount levels (0, 5, 10, 15, 20, 30, 40%). So this is the real value, not a guess.

### 1.7 Strange values

In [ ]:
print("Dates from", orders["order_date"].min(), "to", orders["order_date"].max())
print("Quantity from", orders["quantity"].min(), "to", orders["quantity"].max())
print("Ratings:", sorted(reviews["rating"].dropna().unique().tolist()))
print("Orders with NEGATIVE profit:", (orders["profit_inr"] < 0).sum())

**313 orders have negative profit.** We first thought this was an error, but 1.5 showed the maths is correct. These are real sales where the discount was bigger than the margin, so it is a **business finding, not an error**. We keep them.

### 1.8 Do the tables connect?

In [ ]:
print("Orders with a product that doesn't exist:", (~orders["product_id"].isin(products["product_id"])).sum())
print("Orders without a review:", (~orders["order_id"].isin(reviews["order_id"])).sum())

# we don't have customers.csv, so can we use region_at_order as the customer's region?
regions_per_customer = orders.groupby("customer_id")["region_at_order"].nunique()
print("Customers who ordered from more than 1 region:", (regions_per_customer > 1).sum())

Every customer always orders from the **same region**, so we can use `region_at_order` as the customer's region (we did not get `customers.csv`).

### 1.9 Apply the cleaning

In [ ]:
# 1. remove exact duplicate rows
orders = orders.drop_duplicates()
products = products.drop_duplicates()
reviews = reviews.drop_duplicates()

# 2. fix the 2 wrongly labelled products
products.loc[products["product_id"] == "P011", "category"] = "Beauty"
products.loc[products["product_id"] == "P074", "subcategory"] = "Small Appliances"
products = products.drop(columns="markup")

# 3. fix spelling of sales channel
orders["sales_channel"] = orders["sales_channel"].replace("web site", "Website")

# 4. fill the 1 missing discount using: discount = 1 - revenue / (quantity x price)
orders["discount_pct"] = orders["discount_pct"].fillna(
    (1 - orders["revenue_inr"] / (orders["quantity"] * orders["unit_price_inr"])).round(2))

# 5. label blanks instead of deleting them
orders["customer_id"] = orders["customer_id"].fillna("Unknown")
orders["campaign_id"] = orders["campaign_id"].fillna("No campaign")

# 6. text -> dates
orders["order_date"] = pd.to_datetime(orders["order_date"])
marketing["start_date"] = pd.to_datetime(marketing["start_date"])
marketing["end_date"] = pd.to_datetime(marketing["end_date"])

print("orders:", orders.shape, "| products:", products.shape, "| reviews:", reviews.shape)

### 1.10 Our cleaning log (Problem → Action → Reason)

| # | Problem found | Action taken | Reason |
|---|---|---|---|
| 1 | Hidden character at the start of orders.csv and reviews.csv | Read with `encoding="utf-8-sig"` | Otherwise the `order_id` column name is wrong and joins fail |
| 2 | pandas turns "None" (no complaint) into a blank | Read reviews with `keep_default_na=False` | "None" is real information: 2,640 orders had no complaint |
| 3 | Order O00501 appears twice | Removed 1 copy | Double entry, would add ₹6,706 of fake revenue |
| 4 | Review R00334 appears twice | Removed 1 copy | One review per order |
| 5 | Product P040 appears twice | Removed 1 copy before joining | Would double count all P040 orders |
| 6 | P011 "Hair Care Item" labelled "Electronic" | Changed to Beauty | Name + mark-up (1.92) match Beauty |
| 7 | P074 "Small Appliances Item" labelled "skin care" | Changed to Small Appliances | Name + mark-up (1.47) match Home & Kitchen |
| 8 | Sales channel "web site" | Changed to "Website" | Same channel, different spelling |
| 9 | O00778 missing discount | Calculated it = 0.30 | Worked out from revenue, so not a guess |
| 10 | O00300 missing customer | Kept, customer = "Unknown" | The sale is real |
| 11 | R01201 missing rating | Kept, rating left blank | We don't invent ratings |
| 12 | 1,472 orders with no campaign | Labelled "No campaign" | Blank means no campaign, not an error |
| 13 | 313 orders with negative profit | Kept | Maths is correct, caused by big discounts |
| 14 | Dates stored as text | Converted to dates | Needed for monthly / campaign analysis |
| 15 | customers.csv and delivery.csv not given to us | Used region_at_order and "Late delivery" complaints instead | Each customer always orders from one region |

### 1.11 One clean table

In [ ]:
# join orders + products + reviews using the ID columns
df = orders.merge(products, on="product_id").merge(reviews, on="order_id")
df["margin_%"] = df["profit_inr"] / df["revenue_inr"] * 100

print("Clean table:", df.shape)
df.to_csv("vyapaar_clean.csv", index=False)   # on Colab you can download it from the folder icon
df.head()

✅ **Stage 1 result:** after a few small fixes, the sales and money data can be trusted.

---
# Stage 2: The Business Mystery 💰
### 2.1 Big picture

In [ ]:
total_revenue = df["revenue_inr"].sum()
total_profit = df["profit_inr"].sum()

print("Total revenue: ₹", round(total_revenue / 100000, 2), "lakh")
print("Total profit:  ₹", round(total_profit / 100000, 2), "lakh")
print("Profit margin:", round(total_profit / total_revenue * 100, 1), "%")

### 2.2 Does high revenue mean high profit?

In [ ]:
cat = df.groupby("category")[["revenue_inr", "profit_inr"]].sum()
cat["margin_%"] = cat["profit_inr"] / cat["revenue_inr"] * 100
cat["revenue_share_%"] = cat["revenue_inr"] / cat["revenue_inr"].sum() * 100
cat["profit_share_%"] = cat["profit_inr"] / cat["profit_inr"].sum() * 100
cat = cat.sort_values("revenue_inr", ascending=False)
cat

In [ ]:
# CHART 1
chart1 = cat.reset_index()
fig = px.bar(chart1, x="category", y=["revenue_share_%", "profit_share_%"], barmode="group",
             title="Chart 1: Electronics brings 14.5% of revenue but only 3.2% of profit",
             labels={"value": "% of company total", "variable": ""})
fig.show()

In [ ]:
# How many orders lost money in each category?
df[df["profit_inr"] < 0]["category"].value_counts()

In [ ]:
# Lowest margin subcategories
sub = df.groupby("subcategory")[["revenue_inr", "profit_inr"]].sum()
sub["margin_%"] = sub["profit_inr"] / sub["revenue_inr"] * 100
sub.sort_values("margin_%").head(5)

**So what?**
- **Beauty** is 30% of revenue but **45% of profit**. It is our best category.
- **Electronics** is 14.5% of revenue but only **3.2% of profit** (6.1% margin), and **237 Electronics orders lost money**.
- Why? Electronics is priced at only **1.22 × cost**, while Beauty is 1.92 × cost (see the mark-up check in Stage 1). So Electronics has very little room for discounts.

### 2.3 Are discounts helping?

In [ ]:
disc = df.groupby("discount_pct").agg({"order_id": "count", "quantity": "mean", "revenue_inr": "sum", "profit_inr": "sum"})
disc.columns = ["orders", "avg_quantity", "revenue", "profit"]
disc["profit_per_order"] = disc["profit"] / disc["orders"]
disc["margin_%"] = disc["profit"] / disc["revenue"] * 100
disc

In [ ]:
print("Correlation between discount and quantity:", round(df["discount_pct"].corr(df["quantity"]), 3))

If discounts worked, people would buy **more units** when the discount is bigger. They don't:
- At **0%** discount people buy **1.77** units on average, and at **40%** they buy **1.69**. The correlation is almost **zero**.
- But profit per order drops from **₹1,915** to **−₹33**.

In [ ]:
# CHART 2: margin at each discount level for each category
chart2 = df.pivot_table(index="discount_pct", columns="category", values="margin_%", aggfunc="mean")
chart2.index = chart2.index * 100   # 0.2 -> 20 (%)

fig = px.line(chart2, markers=True,
              title="Chart 2: Electronics loses money from 20% discount; at 40% four categories lose money",
              labels={"discount_pct": "Discount (%)", "value": "Profit margin (%)", "category": "Category"})
fig.add_hline(y=0, line_dash="dash", line_color="black")
fig.show()

In [ ]:
# What if Electronics discounts were capped at 10%?
# (assumption: people still buy the same quantity, which the table above supports)
elec = df[df["category"] == "Electronics"].copy()
elec["new_discount"] = elec["discount_pct"].clip(upper=0.10)
elec["new_profit"] = elec["quantity"] * elec["unit_price_inr"] * (1 - elec["new_discount"]) - elec["cost_inr"]

print("Electronics profit now: ₹", round(elec["profit_inr"].sum() / 100000, 2), "lakh")
print("With a 10% cap:         ₹", round(elec["new_profit"].sum() / 100000, 2), "lakh")

In [ ]:
# How much did we give away in discounts?
full_price_sales = (df["quantity"] * df["unit_price_inr"]).sum()
print("Discounts given in the year: ₹", round((full_price_sales - total_revenue) / 100000, 2), "lakh")

# Deep discounts (30% and 40%)
deep = df[df["discount_pct"] >= 0.30]
print("Orders at 30-40% discount:", len(deep), "=", round(len(deep) / len(df) * 100), "% of orders")
print("Their share of profit:", round(deep["profit_inr"].sum() / total_profit * 100, 1), "%")

In [ ]:
# Which category gets the most deep discounts? (% of its orders at 30-40% off)
(deep["category"].value_counts() / df["category"].value_counts() * 100).sort_values(ascending=False)

**So what?**
- The data **does not support** the current discount strategy. Bigger discounts don't make people buy more, they just lower our margin.
- We gave away **₹28 lakh** in discounts, more than half of our total profit (₹48 lakh).
- Orders at 30-40% off are 18% of orders but only **3.5% of profit**.
- Capping Electronics discounts at 10% would almost **double** Electronics profit (₹1.55 → ₹3.06 lakh).
- **55% of Fashion orders** are at 30-40% off, so Fashion is basically always on sale.

### 2.4 Customers
**Our definition of a high-value customer:** a customer in the **top 10% by total profit** in 2025. We use profit and not revenue, because revenue can hide loss-making orders.

In [ ]:
known = df[df["customer_id"] != "Unknown"]
cust = known.groupby("customer_id").agg({"order_id": "count", "revenue_inr": "sum", "profit_inr": "sum"})
cust.columns = ["orders", "revenue", "profit"]

cutoff = cust["profit"].quantile(0.90)
high_value = cust[cust["profit"] >= cutoff]

print("Total customers:", len(cust))
print("High-value cut-off: ₹", round(cutoff), "profit")
print("High-value customers:", len(high_value))
print("Their average orders:", round(high_value["orders"].mean(), 1), "vs everyone:", round(cust["orders"].mean(), 1))
print("Their share of profit:", round(high_value["profit"].sum() / cust["profit"].sum() * 100, 1), "%")

In [ ]:
print("Customers who bought more than once:", round((cust["orders"] > 1).mean() * 100, 1), "%")
print("Customers who bought 5 or more times:", round((cust["orders"] >= 5).mean() * 100, 1), "%")

In [ ]:
# When did each customer buy for the first time?
first_order = known.groupby("customer_id")["order_date"].min()
first_order.dt.quarter.value_counts().sort_index()

**So what?**
- **98.8%** of customers come back, so Vyapaar is a **repeat-customer business**. Keeping customers happy is the most important thing.
- The **60 high-value customers** order about **9 times** a year and bring **23.5% of profit**.
- **490 of 599** customers first bought in Q1. Only **15 new customers** joined in the second half of the year (Q3 + Q4). We look at this again in the marketing part.

---
# Stage 3: Customer & Operations Mystery 🚚
*We did not get `delivery.csv`, so we use the review complaint **"Late delivery"** as our sign of a delivery problem.*

### 3.1 Which complaint hurts the most?

In [ ]:
comp = df.groupby("complaint_category").agg({"order_id": "count", "rating": "mean", "escalated": "mean"})
comp.columns = ["orders", "avg_rating", "escalation_rate"]
comp["escalation_rate"] = comp["escalation_rate"] * 100
comp.sort_values("orders", ascending=False)

- **Late delivery** is the most common complaint (695 orders).
- It is also the worst one: average rating **2.43** and **48%** of these customers escalate. Product complaints are only escalated 7-14% of the time.

### 3.2 Where is it happening?

In [ ]:
df["late_delivery"] = df["complaint_category"] == "Late delivery"

region = df.groupby("region_at_order").agg({"order_id": "count", "late_delivery": "mean", "escalated": ["mean", "sum"], "rating": "mean"})
region.columns = ["orders", "late_delivery_%", "escalation_%", "escalations", "avg_rating"]
region["late_delivery_%"] = region["late_delivery_%"] * 100
region["escalation_%"] = region["escalation_%"] * 100
region["share_of_escalations_%"] = region["escalations"] / region["escalations"].sum() * 100
region

In [ ]:
# CHART 3
chart3 = region.reset_index()
fig = px.bar(chart3, x="region_at_order", y=["late_delivery_%", "escalation_%"], barmode="group", text_auto=".1f",
             title="Chart 3: East has 3x the late deliveries and about 4x the escalations of other regions",
             labels={"region_at_order": "Region", "value": "% of orders", "variable": ""})
fig.show()

In [ ]:
# Is it just one bad month? (% late-delivery complaints by month)
df["month"] = df["order_date"].dt.month
(df.pivot_table(index="month", columns="region_at_order", values="late_delivery", aggfunc="mean") * 100).round(0)

In [ ]:
# Is it one sales channel?
(df.pivot_table(index="region_at_order", columns="sales_channel", values="late_delivery", aggfunc="mean") * 100).round(0)

**What we found:**
- **38.5%** of East orders get a late-delivery complaint. In the other regions it is only **12-13%**.
- East is **19% of orders** but **48% of all escalations**, and its rating is **3.23** vs about 3.9 elsewhere.
- It is high in **every month** and **every channel**, so it is a permanent East delivery problem, not a one-time thing.

### 3.3 Do unhappy customers stop buying?

In [ ]:
known = df[df["customer_id"] != "Unknown"]
per_region = known.groupby("region_at_order").agg({"order_id": "count", "customer_id": "nunique"})
per_region["orders_per_customer"] = per_region["order_id"] / per_region["customer_id"]
per_region

In [ ]:
# Small thing we noticed: Store orders also get late-delivery complaints
store = df[df["sales_channel"] == "Store"]
print("Store orders with a late-delivery complaint:", store["late_delivery"].sum(), "out of", len(store))

**Honest result:** East customers still buy about **6.5 times** a year, which is close to the other regions (6.6-6.9). So with one year of data we **cannot prove** late delivery makes customers leave.

**Why it still matters:**
1. Every escalation costs customer-service time and money, and East creates almost half of them.
2. A 3.2 rating is visible to new customers on apps and marketplaces.
3. Loyal customers may stay only until a competitor delivers faster.

*Also:* 179 **Store** orders have late-delivery complaints, which is strange for an in-store purchase. We need `delivery.csv` to check this.

**Recommendation:** audit the East delivery partner and promised delivery dates, and get `delivery.csv` to find the exact cause.

---
# Stage 4: OUR HIDDEN MYSTERY 🕵️
## "Where does the money go, and is marketing worth it?"

**Question:** The orders show ₹48 lakh profit, but that ignores marketing. After paying for the 18 campaigns, how much is left? And can we prove the campaigns bring in sales?

### 4.1 Where does the money go?

In [ ]:
full_price_sales = (df["quantity"] * df["unit_price_inr"]).sum()
revenue = df["revenue_inr"].sum()
discounts = full_price_sales - revenue
product_cost = df["cost_inr"].sum()
gross_profit = df["profit_inr"].sum()
marketing_spend = marketing["spend_inr"].sum()
left = gross_profit - marketing_spend

print("Sales at full price:  ₹", round(full_price_sales / 100000, 2), "lakh")
print("- Discounts:          ₹", round(discounts / 100000, 2), "lakh")
print("= Revenue:            ₹", round(revenue / 100000, 2), "lakh")
print("- Product cost:       ₹", round(product_cost / 100000, 2), "lakh")
print("= Gross profit:       ₹", round(gross_profit / 100000, 2), "lakh")
print("- Marketing spend:    ₹", round(marketing_spend / 100000, 2), "lakh")
print("= Left after marketing: ₹", round(left / 100000, 2), "lakh")
print()
print("Marketing spend as % of gross profit:", round(marketing_spend / gross_profit * 100), "%")

In [ ]:
# CHART 4
money = pd.DataFrame({
    "step": ["Sales at full price", "Discounts", "Revenue", "Product cost", "Gross profit", "Marketing spend", "Left after marketing"],
    "lakh": [full_price_sales, discounts, revenue, product_cost, gross_profit, marketing_spend, left],
    "type": ["Money in", "Money out", "Money in", "Money out", "Money in", "Money out", "Money in"],
})
money["lakh"] = (money["lakh"] / 100000).round(1)

fig = px.bar(money, x="step", y="lakh", color="type", text_auto=True,
             color_discrete_map={"Money in": "royalblue", "Money out": "indianred"},
             title="Chart 4: Marketing (₹47.0 lakh) eats 98% of gross profit (₹48.0 lakh)",
             labels={"step": "", "lakh": "₹ lakh", "type": ""})
fig.show()

Almost **all** of the profit goes on marketing, and this is before delivery costs, salaries and rent (which we don't have). So: **is marketing actually bringing in the sales?**

### 4.2 Were "campaign orders" placed during the campaign?

In [ ]:
tagged = df[df["campaign_id"] != "No campaign"]
tagged = tagged.merge(marketing[["campaign_id", "start_date", "end_date"]], on="campaign_id")

def timing(row):
    if row["order_date"] < row["start_date"]:
        return "Before campaign started"
    elif row["order_date"] > row["end_date"]:
        return "After campaign ended"
    else:
        return "During campaign"

tagged["timing"] = tagged.apply(timing, axis=1)
print(tagged["timing"].value_counts())
print()
print((tagged["timing"].value_counts(normalize=True) * 100).round(1))

In [ ]:
# CHART 5
chart5 = tagged.groupby(["campaign_id", "timing"]).size().reset_index(name="orders")
fig = px.bar(chart5, x="campaign_id", y="orders", color="timing",
             color_discrete_map={"Before campaign started": "indianred", "During campaign": "royalblue", "After campaign ended": "lightgrey"},
             title="Chart 5: Only 11% of campaign orders were placed while the campaign was running",
             labels={"campaign_id": "Campaign", "orders": "Orders tagged to the campaign", "timing": ""})
fig.show()

In [ ]:
# Does a campaign that spends more get more orders?
per_campaign = tagged.groupby("campaign_id")["order_id"].count().reset_index(name="tagged_orders")
per_campaign = per_campaign.merge(marketing[["campaign_id", "channel", "spend_inr", "conversions"]], on="campaign_id")
print("Correlation between spend and tagged orders:", round(per_campaign["spend_inr"].corr(per_campaign["tagged_orders"]), 2))
per_campaign

In [ ]:
# Marketing says it got this many conversions... how many customers do we have?
known = df[df["customer_id"] != "Unknown"]
first_order = known.groupby("customer_id")["order_date"].min()
h2 = marketing[marketing["start_date"] >= "2025-07-01"]

print("Conversions reported by marketing:", marketing["conversions"].sum())
print("Total customers in our orders:", known["customer_id"].nunique())
print()
print("Spend on campaigns started in Jul-Dec: ₹", round(h2["spend_inr"].sum() / 100000, 2), "lakh")
print("Conversions they reported:", h2["conversions"].sum())
print("NEW customers who first bought in Jul-Dec:", (first_order >= "2025-07-01").sum())

### Evidence
1. **Marketing (₹47.0 lakh) = 98% of gross profit (₹48.0 lakh).** Less than **₹1 lakh** is left.
2. **48%** of "campaign" orders happened **before the campaign even started**, and only **11%** during it. That is impossible if the tag really means "this campaign brought the sale".
3. Spending more doesn't bring more tagged orders (correlation only 0.19).
4. Marketing reports **4,309 conversions**, but we only have **599 customers**. Campaigns in Jul-Dec spent **₹20.75 lakh**, but only **15 new customers** joined in that time.

### Why management should care
After the product itself, **marketing is the biggest cost**, and right now **nobody can prove it works**. Since almost all customers are repeat buyers, many "conversions" may be people who would have bought anyway.

*Assumptions:* we treat all campaign spend as 2025 cost (2 campaigns run a few weeks into Jan 2026), and we don't know exactly how marketing defines a "conversion".

### Recommendation
1. **Fix tracking:** one coupon code or link per campaign, and don't allow a campaign tag on an order outside the campaign dates.
2. Count a conversion only as a **new customer's first order**.
3. **Test it:** pause one channel in one region for a month and see if sales drop.
4. Cut the weakest channel first (see the Bonus below).

---
# Bonus: Marketing Mystery 📣
Here we use marketing's **own numbers** (remember the doubts above).

In [ ]:
ch = marketing.groupby("channel")[["spend_inr", "leads", "conversions"]].sum()
ch["conversion_rate_%"] = ch["conversions"] / ch["leads"] * 100
ch["cost_per_conversion"] = ch["spend_inr"] / ch["conversions"]
ch["share_of_spend_%"] = ch["spend_inr"] / ch["spend_inr"].sum() * 100
ch["share_of_conversions_%"] = ch["conversions"] / ch["conversions"].sum() * 100
ch.sort_values("cost_per_conversion")

In [ ]:
known = df[df["customer_id"] != "Unknown"]
print("Average profit per order: ₹", round(df["profit_inr"].mean()))
print("Average profit per customer per year: ₹", round(known.groupby("customer_id")["profit_inr"].sum().mean()))

**So what?**
- **Outdoor** costs **₹6,587 per conversion**, about **10 times** Affiliate (₹642). It uses **15.6%** of the budget but brings only **2.6%** of conversions.
- **More leads ≠ success.** Influencer and Social Media bring lots of leads, but only 6-8% convert. **Email** converts the best (14%).
- ⚠️ **This is NOT ROI.** For ROI we would need the profit each campaign really caused, and 4.2 showed we can't get that from this data. We can only say that one Outdoor conversion costs more than the profit from one order (₹1,199).
- **Recommendation:** move Outdoor money to Email and Affiliate.

---
# Management Summary 🎯

### Finding 1: Electronics sells but barely makes money
- **Evidence:** 14.5% of revenue but only 3.2% of profit (6.1% margin). 237 Electronics orders lost money. It goes negative at a 20% discount. *(Chart 1, Chart 2)*
- **Implication:** we spend discounts on a category that can't afford them. Beauty (45% of profit) is the real profit engine.
- **Action:** cap Electronics discounts at **10%** (about **+₹1.5 lakh** profit a year), renegotiate supplier costs, and promote Beauty more.

### Finding 2: Big discounts don't increase sales
- **Evidence:** 1.77 units per order at 0% off vs 1.69 at 40% off. Profit per order falls from ₹1,915 to −₹33. ₹28 lakh was given away in discounts. *(Chart 2)*
- **Implication:** the discount strategy is not supported by the data.
- **Action:** set a **maximum discount per category** (no 40% discounts at all; Electronics max 10%) and test any big sale on a small group first.

### Finding 3: East has a delivery problem
- **Evidence:** 38.5% of East orders have a late-delivery complaint vs 12-13% elsewhere. East is 19% of orders but 48% of escalations, and its rating is 3.2 vs 3.9. *(Chart 3)*
- **Implication:** 99% of our customers come back, so customer experience is our biggest asset. Escalations cost money and low ratings put off new customers.
- **Action:** audit the East delivery partner within 30 days, set an on-time target for East, and get `delivery.csv` analysed.

### OUR HIDDEN MYSTERY: Marketing eats 98% of profit and we can't prove it works
- **Question:** after marketing, how much profit is left, and do campaigns really bring in sales?
- **Evidence:** ₹47.0 lakh of marketing vs ₹48.0 lakh gross profit. Only 11% of campaign orders were placed during the campaign (48% before it started). 4,309 "conversions" reported but only 599 customers. *(Chart 4, Chart 5)*
- **Implication:** our biggest controllable cost is not being measured.
- **Recommendation:** fix campaign tracking, count only new customers as conversions, test by pausing a channel, and move Outdoor money (₹6,587 per conversion) to Email/Affiliate.

### Limitations
- We did not have `customers.csv` or `delivery.csv`.
- "Profit" here = revenue − product cost only. Delivery, salaries and rent are not in the data.
- Only one year of data.